# Assignment 2: Comparing Differential Evolution and Cross Entropy Method

**Course:** Neural Networks and Genetic Algorithms - CS410.Q2  
**Student ID:** 23520108  
**Objective:** Implementation, performance comparison, and visualization of two optimization algorithms: Differential Evolution (DE) and Cross Entropy Method (CEM)

## 1. Library Initialization and Utilities

Import necessary libraries for numerical computation, visualization, and animation generation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy import stats
from IPython.display import display, Markdown, HTML

MSSV = 23520108 

# Function to return seed based on index
def get_seed(i):
    return MSSV + i

In [ ]:
# Objective Functions Implementation
def sphere(x):
    return np.sum(x**2)

def rastrigin(x):
    return 10 * len(x) + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2.0 + (1 - x[:-1])**2.0)

def ackley(x):
    d = len(x)
    sum_sq = np.sum(x**2)
    sum_cos = np.sum(np.cos(2 * np.pi * x))
    term1 = -20 * np.exp(-0.2 * np.sqrt(sum_sq / d))
    term2 = -np.exp(sum_cos / d)
    return term1 + term2 + 20 + np.e

def griewank(x):
    sum_sq = np.sum(x**2) / 4000.0
    prod_cos = np.prod(np.cos(x / np.sqrt(np.arange(1, len(x) + 1))))
    return sum_sq - prod_cos + 1.0

# Define benchmark functions with standard search space bounds
FUNCTIONS = {
    'Sphere': {'func': sphere, 'bound': [-5.12, 5.12], 'opt_val': 0.0},
    'Rastrigin': {'func': rastrigin, 'bound': [-5.12, 5.12], 'opt_val': 0.0},
    'Rosenbrock': {'func': rosenbrock, 'bound': [-5.0, 10.0], 'opt_val': 0.0},
    'Ackley': {'func': ackley, 'bound': [-32.0, 32.0], 'opt_val': 0.0},
    'Griewank': {'func': griewank, 'bound': [-600.0, 600.0], 'opt_val': 0.0}
}

## 2. Objective Functions Definition

Implementation of 5 popular benchmark functions: Sphere, Rastrigin, Rosenbrock, Ackley, and Griewank. Each function has different search space bounds and varying levels of optimization difficulty.

In [ ]:
# Differential Evolution (DE) Algorithm
def differential_evolution(func, bounds, popsize, max_evals, seed, return_history=False, return_pop_history=False):
    np.random.seed(seed)
    d = len(bounds)
    
    # Initialize population uniformly within bounds
    pop = np.random.uniform(bounds[:, 0], bounds[:, 1], size=(popsize, d))
    fitness = np.apply_along_axis(func, 1, pop)
    evals = popsize
    
    best_idx = np.argmin(fitness)
    best_fit = fitness[best_idx]
    
    # DE parameters
    F = 0.8  # Mutation scale factor
    CR = 0.9  # Crossover rate
    
    history = [(evals, best_fit)]
    pop_history = [pop.copy()] if return_pop_history else []
    
    while evals < max_evals:
        for i in range(popsize):
            if evals >= max_evals:
                break
                
            # Select 3 random individuals different from i
            idxs = [idx for idx in range(popsize) if idx != i]
            a, b, c = pop[np.random.choice(idxs, 3, replace=False)]
            
            # Mutation step: create mutant vector
            mutant = np.clip(a + F * (b - c), bounds[:, 0], bounds[:, 1])
            cross_points = np.random.rand(d) < CR
            
            # Ensure at least one dimension is selected
            if not np.any(cross_points):
                cross_points[np.random.randint(0, d)] = True
            
            # Crossover step: create trial vector
            trial = np.where(cross_points, mutant, pop[i])
            
            # Evaluate fitness
            f_trial = func(trial)
            evals += 1
            
            # Selection step: replace if better
            if f_trial < fitness[i]:
                pop[i] = trial
                fitness[i] = f_trial
                if f_trial < best_fit:
                    best_fit = f_trial
                    
        # Record history after each generation
        if return_history:
            history.append((evals, best_fit))
        if return_pop_history:
            pop_history.append(pop.copy())
            
    if return_history or return_pop_history:
        if return_history and return_pop_history:
            return best_fit, history, pop_history
        elif return_history:
            return best_fit, history
        return best_fit, pop_history
    return best_fit

## 3. Differential Evolution (DE) Algorithm Implementation

The DE algorithm employs three main operations:
- **Mutation:** Create a mutant vector from three randomly selected individuals
- **Crossover:** Create a trial vector by combining genes from the mutant and parent
- **Selection:** Keep the individual if its fitness is better than the current one

Parameters: F (mutation scale factor) = 0.8, CR (crossover probability) = 0.9

In [ ]:
# Cross Entropy Method (CEM) Algorithm
def cross_entropy_method(func, bounds, popsize, max_evals, seed, return_history=False, return_pop_history=False):
    np.random.seed(seed)
    d = len(bounds)
    num_elites = max(2, int(popsize * 0.2))  # Select top 20% as elite individuals
    
    # Initialize Gaussian distribution model
    mu = np.random.uniform(bounds[:, 0], bounds[:, 1])
    sigma = (bounds[:, 1] - bounds[:, 0]) / 2.0
    
    evals = 0
    best_fit = np.inf
    
    history = []
    pop_history = []
    
    while evals < max_evals:
        # Adjust population size if near evaluation budget limit
        current_popsize = min(popsize, max_evals - evals)
        
        # Sample population from Gaussian distribution
        pop = np.random.normal(mu, sigma, size=(current_popsize, d))
        pop = np.clip(pop, bounds[:, 0], bounds[:, 1])
        
        if return_pop_history:
            pop_history.append(pop.copy())
            
        # Evaluate population
        fitness = np.apply_along_axis(func, 1, pop)
        evals += current_popsize
        
        current_best_fit = np.min(fitness)
        if current_best_fit < best_fit:
            best_fit = current_best_fit
            
        if return_history:
            history.append((evals, best_fit))
            
        # Select elite individuals
        elite_idxs = np.argsort(fitness)[:min(num_elites, current_popsize)]
        elites = pop[elite_idxs]
        
        # Update distribution parameters
        mu = np.mean(elites, axis=0)
        sigma = np.std(elites, axis=0) + 1e-5
        
    if return_history or return_pop_history:
        if return_history and return_pop_history:
            return best_fit, history, pop_history
        elif return_history:
            return best_fit, history
        return best_fit, pop_history
    return best_fit

## 4. Cross Entropy Method (CEM) Algorithm Implementation

The CEM is a probabilistic model-based optimization method consisting of:
- **Distribution Initialization:** A Gaussian model initially covering the entire search space
- **Population Sampling:** Generate individuals by sampling from the current Gaussian distribution
- **Elite Selection:** Select the top 20% of individuals with best fitness
- **Model Update:** Adjust mean and standard deviation based on elite individuals

This iterative process continues until the evaluation budget is exhausted.

In [ ]:
# Experimental Setup and Execution
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # Suppress unnecessary warnings

dims = [2, 10]
popsizes = [8, 16, 32, 64, 128]
num_runs = 10

experiment_results = []
convergence_data = {} 

# Large-scale experimental loop
# Expected runtime: approximately 1-2 minutes depending on machine speed
for func_name, info in FUNCTIONS.items():
    convergence_data[func_name] = {}
    func = info['func']
    for d in dims:
        bounds = np.array([info['bound']] * d)
        max_evals = 2000 if d == 2 else 10000
        convergence_data[func_name][d] = {}
        for N in popsizes:
            de_bests = []
            cem_bests = []
            
            de_histories = []
            cem_histories = []
            
            for i in range(num_runs):
                seed = get_seed(i)
                # Run Differential Evolution
                de_best, de_hist = differential_evolution(func, bounds, N, max_evals, seed, return_history=True)
                de_bests.append(de_best)
                de_histories.append(de_hist)
                
                # Run Cross Entropy Method
                cem_best, cem_hist = cross_entropy_method(func, bounds, N, max_evals, seed, return_history=True)
                cem_bests.append(cem_best)
                cem_histories.append(cem_hist)
                
            # Store history for later plotting
            convergence_data[func_name][d][N] = {
                'DE_hists': de_histories,
                'CEM_hists': cem_histories
            }
                
            de_mean = np.mean(de_bests)
            de_std = np.std(de_bests)
            
            cem_mean = np.mean(cem_bests)
            cem_std = np.std(cem_bests)
            
            # Statistical significance testing using Independent t-test
            t_stat, p_value = stats.ttest_ind(de_bests, cem_bests)
            
            if p_value < 0.05:
                if de_mean < cem_mean:
                    winner = 'DE'
                else:
                    winner = 'CEM'
            else:
                winner = 'Tie'
                
            experiment_results.append({
                'Function': func_name,
                'd': d,
                'N': N,
                'DE_Mean': de_mean,
                'DE_Std': de_std,
                'CEM_Mean': cem_mean,
                'CEM_Std': cem_std,
                'Winner': winner
            })

df_results = pd.DataFrame(experiment_results)

## 5. Experimental Design and Execution

Conducting a comprehensive comparative study between DE and CEM:
- **Functions:** 5 benchmark functions (Sphere, Rastrigin, Rosenbrock, Ackley, Griewank)
- **Dimensions (d):** 2 and 10
- **Population Size (N):** 8, 16, 32, 64, 128
- **Runs per Configuration:** 10 independent runs

Statistical significance is determined using Independent t-test (p < 0.05)

In [ ]:
# Results Summary in Markdown Table Format
md_lines = ["| Function | N | d | DE_Mean(std) | CEM_Mean(std) | Winner |", "|---|---|---|---|---|---|"]

for _, row in df_results.iterrows():
    de_str = f"{row['DE_Mean']:.4e} ({row['DE_Std']:.4e})"
    cem_str = f"{row['CEM_Mean']:.4e} ({row['CEM_Std']:.4e})"
    
    # Highlight winning algorithm in bold
    if row['Winner'] == 'DE':
        de_str = f"**{de_str}**"
    elif row['Winner'] == 'CEM':
        cem_str = f"**{cem_str}**"

    md_lines.append(f"| {row['Function']} | {row['N']} | {row['d']} | {de_str} | {cem_str} | {row['Winner']} |")

display(Markdown("\n".join(md_lines)))

## 6. Results and Statistical Analysis

Summary table of comparison results across all configurations:
- **DE_Mean(std):** Mean fitness value and standard deviation for DE
- **CEM_Mean(std):** Mean fitness value and standard deviation for CEM
- **Winner:** The algorithm with superior performance (shown in bold). "Tie" indicates no statistically significant difference

In [ ]:
# Convergence Curve Visualization
def plot_convergence(func_name, d, N):
    d_hists = convergence_data[func_name][d][N]['DE_hists']
    c_hists = convergence_data[func_name][d][N]['CEM_hists']
    
    def extract_stats(hists):
        # Extract evaluation counts and fitness values from all runs
        evals_arr = np.array([x[0] for x in hists[0]])
        vals_arr = np.array([[x[1] for x in run_hist] for run_hist in hists])
        mean_vals = np.mean(vals_arr, axis=0)
        std_vals = np.std(vals_arr, axis=0)
        return evals_arr, mean_vals, std_vals

    de_evals, de_mean, de_std = extract_stats(d_hists)
    cem_evals, cem_mean, cem_std = extract_stats(c_hists)

    plt.figure(figsize=(10, 6))
    
    # Plot DE convergence curve with shaded standard deviation band
    plt.plot(de_evals, de_mean, label='DE Mean Fitness', color='blue')
    plt.fill_between(de_evals, de_mean - de_std, de_mean + de_std, alpha=0.3, color='blue')
    
    # Plot CEM convergence curve with shaded standard deviation band
    plt.plot(cem_evals, cem_mean, label='CEM Mean Fitness', color='red')
    plt.fill_between(cem_evals, cem_mean - cem_std, cem_mean + cem_std, alpha=0.3, color='red')
    
    plt.yscale('log')
    plt.title(f'Convergence Graph (d={d}, N={N}) - Function: {func_name}')
    plt.xlabel('Fitness Evaluations')
    plt.ylabel('Best Fitness Value')
    plt.legend()
    plt.grid(True)
    plt.show()

# Demonstrate convergence plot with Rastrigin as example
plot_convergence('Rastrigin', d=10, N=32)

## 7. Convergence Analysis

Plotting convergence curves for each configuration:
- **X-axis:** Number of objective function evaluations
- **Y-axis:** Best fitness value found (logarithmic scale)
- **Shaded Region:** ±1 standard deviation band across 10 runs

Example shown: Rastrigin function, 10-dimensional, population size 32

In [ ]:
import os

# Create directory to store GIF files
os.makedirs("GIF_Results", exist_ok=True)

# Function to generate animation GIF for algorithms
def make_gif_animation_fixed(algo_name, func_name):
    d = 2
    N = 32
    max_evals = 2000
    seed = MSSV
    
    info = FUNCTIONS[func_name]
    func = info['func']
    bounds = np.array([info['bound']] * d)
    
    # Run algorithm and get population history
    if algo_name == 'DE':
        _, best_hists, pop_hists = differential_evolution(func, bounds, N, max_evals, seed, return_history=True, return_pop_history=True)
    else:
        _, best_hists, pop_hists = cross_entropy_method(func, bounds, N, max_evals, seed, return_history=True, return_pop_history=True)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # Create contour plot of objective function
    x = np.linspace(bounds[0][0], bounds[0][1], 100)
    y = np.linspace(bounds[1][0], bounds[1][1], 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    for i in range(100):
        for j in range(100):
            Z[i, j] = func(np.array([X[i, j], Y[i, j]]))
            
    CS = ax.contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.6)
    
    # Mark global optimum location
    if func_name == 'Rosenbrock':
        opt_x, opt_y = 1.0, 1.0
    else:
        opt_x, opt_y = 0.0, 0.0
    
    ax.scatter(opt_x, opt_y, marker='D', color='red', s=100, label='Global Optimum')
    scat = ax.scatter([], [], color='orange', edgecolor='black', s=40)
    ax.legend(loc='upper right')
    
    # Animation update function
    def update(frame):
        pop = pop_hists[frame]
        scat.set_offsets(pop) 
        ax.set_title(f'{algo_name} on {func_name} - Generation {frame+1}\nEvals: {best_hists[frame][0]} | Best: {best_hists[frame][1]:.4e}')
        return scat,
        
    anim = animation.FuncAnimation(fig, update, frames=len(pop_hists), interval=150, blit=True)
    
    # Save animation as GIF file
    filename = f'GIF_Results/{algo_name}_{func_name}_d2_N32.gif'
    anim.save(filename, writer='pillow', fps=6)
    plt.close()
    
    return filename

# Generate 10 GIF files (5 functions × 2 algorithms)
print("Generating 10 GIF files for all functions and algorithms... (Please wait a few minutes)")
gif_paths = []
for func_name in FUNCTIONS.keys():
    print(f"  -> Generating GIF for function {func_name}...")
    de_gif = make_gif_animation_fixed('DE', func_name)
    cem_gif = make_gif_animation_fixed('CEM', func_name)
    gif_paths.extend([de_gif, cem_gif])

print("All GIF files generated successfully! You can download the 'GIF_Results' directory for submission.")

## 8. Animation Generation (GIF)

Creating 10 GIF files (5 functions × 2 algorithms) for dynamic visualization:
- **Each Frame:** One generation of the algorithm
- **Background:** 2D contour map of the objective function
- **Orange Points:** Current population
- **Red Point:** Global optimum location
- **Title:** Displays number of function evaluations and best fitness value

All GIF files are saved in the `GIF_Results/` directory

In [ ]:
display(Markdown("### DE Animation"))
display(HTML(f'<img src="{de_gif}" alt="DE" width="400">'))

display(Markdown("### CEM Animation"))
display(HTML(f'<img src="{cem_gif}" alt="CEM" width="400">'))


### DE Animation

### CEM Animation

## 9. Animation Display

View the final animations generated (DE and CEM on Griewank function, 2-dimensional)